In [ ]:
%pip install requests beautifulsoup4 playwright nest_asyncio

UsageError: Line magic function `%playwright` not found.


In [2]:
import os
import asyncio
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from playwright.async_api import async_playwright
import nest_asyncio
import re

# Allow nested async loops in Jupyter
nest_asyncio.apply()

# Table of Contents URL (update if needed)
toc_url = "https://practicalguidetoevil.wordpress.com/table-of-contents/"
output_dir = Path(os.getcwd())  # Save folders in current working directory

In [ ]:
def get_books_and_links():
    response = requests.get(toc_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    books = {}
    current_book = None

    # Go through all tags in order
    tags = soup.find_all(["h2", "ul"])
    i = 0
    while i < len(tags):
        tag = tags[i]
        
        if tag.name == "h2" and "Book" in tag.get_text():
            current_book = tag.get_text(strip=True)
            books[current_book] = []

            # Check if the next tag is a <ul> and collect its links
            if i + 1 < len(tags) and tags[i + 1].name == "ul":
                ul = tags[i + 1]
                for li in ul.find_all("li", recursive=False):  # Avoid nested <li><ul>
                    a = li.find("a", href=True)
                    if a:
                        title = a.get_text(strip=True)
                        href = a["href"]
                        books[current_book].append((title, href))
                i += 1  # Skip the ul we just processed

        i += 1

    return books

books = get_books_and_links()

# Preview result
for book, chapters in books.items():
    print(f"\n📘 {book} ({len(chapters)} chapters)")
    for title, _ in chapters:  # Show first 3 chapters
        print(f"  - {title}")


📘 Book 1 (30 chapters)
  - Prologue
  - Chapter 1: Knife
  - Chapter 2: Invitation
  - Chapter 3: Party
  - Chapter 4: Name
  - Chapter 5: Role
  - Chapter 6: Aspect
  - Chapter 7: Sword
  - Chapter 8: Introduction
  - Chapter 9: Claimant
  - Chapter 10: Menace
  - Chapter 11: Sucker Punch
  - Chapter 12: Squire
  - Chapter 13: Order
  - Chapter 14: Villain
  - Chapter 15: Company
  - Chapter 16: Game
  - Chapter 17: Set
  - Chapter 18: Match
  - Chapter 19: Pivot
  - Chapter 20: Rise
  - Chapter 21: Fall
  - Chapter 22: All According To
  - Chapter 23: Morok’s Plan
  - Chapter 24: Aisha’s Plan
  - Chapter 25: Snatcher’s Plan
  - Chapter 26: Juniper’s Plan
  - Chapter 27: Callow’s Plan
  - Chapter 28: Win Condition
  - Epilogue

📘 Book 2 (125 chapters)
  - Prologue
  - Prologue
  - Chapter 1: Supply
  - Chapter 2: Demand
  - Chapter 3: Cost
  - Heroic Interlude: Balestra
  - Chapter 4: Return
  - Chapter 5: Recognition
  - Chapter 6: Rapport
  - Chapter 7: Reception
  - Chapter 8: Rev

In [ ]:
def sanitize_filename(title, index):
    title = re.sub(r'[^\w\s-]', '', title)  # Remove special characters
    title = re.sub(r'\s+', '_', title.strip())  # Replace spaces with underscores
    return f"Chapter_{index:02d}_{title}.pdf"